## Lesson 3: Scaling to 10 Million+ Rows (Python Optimization)
Now that you’ve mastered the "Watchdog" logic, we hit a new wall. If your order_discounts table grows to 10 million rows, pd.read_sql_query will hang, eat all your RAM, and crash your script.

A 100x Analyst knows that Memory is a Finite Resource.

#### 1. The Scenario
The "SPRING20" exploit happened during a massive holiday sale. Your database is flooded. You need to run your fraud analysis, but the CSV/Table is too large to load into a standard Pandas DataFrame.

#### 2. The Logic: "Chunking" vs. "Vectorization"
**Junior Approach:** Buy a bigger server with more RAM (Expensive/Slow).

**100x Approach:** Process the data in Chunks or use Vectorized operations directly in the database.

#### 3. The Execution (Python Chunking)
Instead of pulling everything, we process the data in "Bites."

In [ ]:
import pandas as pd
import sqlite3

def heavy_duty_monitor():
    conn = sqlite3.connect('big_data.db')
    chunk_size = 100000 # 100k rows at a time
    total_leakage = 0
    
    # Using 'chunksize' returns an iterator, not a massive object
    query = "SELECT discount_amount FROM order_discounts WHERE created_at >= ..."
    
    for chunk in pd.read_sql_query(query, conn, chunksize=chunk_size):
        # Process each bite-sized piece
        total_leakage += chunk['discount_amount'].sum()
        
        if total_leakage > 10000:
            print("Threshold crossed mid-process!")
            break 

    return total_leakage

#### 4. The Solution Provisioning: SQL Pushdown
Actually, the truest 100x move is realizing you shouldn't use Python for the math at all if the DB can do it. This is called "Pushing down the logic."

- **Bad:** Pull 10M rows into Python -> Sum them.
- **100x:** Let SQL (which is optimized for this) sum the 10M rows -> Pull 1 row (the total) into Python.

#### Your 10-Minute Technical Challenge
I want you to think about Data Integrity.

If you are "Chunking" a 10-million row file to find "User Abuse," and a specific user's transactions are split across two different chunks (50 in Chunk A, 50 in Chunk B), your script might miss them because neither chunk alone hits the "100 usages" threshold.

How would you solve this "Split Data" problem without loading the whole file into RAM? (Hint: Think about how you would sort the data or use a 'Dictionary' to keep a running tally).

Give me your logic, and tomorrow we dive into Day 4: Communicating Results with Data Storytelling (The "So What?" Principle). Would you like to tackle the "Split User" logic first?

#### 1x Analyst:
To solve this issue, I would order by user id to make sure that each user is captured in corresponding chunks, Or better still, make my sql query do the aggregarion for me.

#### 100x Analyst:
You are thinking exactly like a senior engineer. You’ve identified the two primary ways to solve the "Distributed State" problem: Pre-Sorting (at the application layer) vs. Delegation (at the database layer).

**The 100x Verdict:** What is the Optimal Solution?
The optimal solution is almost always Delegation (The SQL Pushdown).

In a professional environment, moving 10 million rows across a network from a Database to a Python script is a "bottleneck" (it’s slow and expensive). Letting the database engine—which is purpose-built for massive aggregations—do the work is the "100x" move.

However, if you are working with a CSV file (no database) or a frozen API, you must use the Python Dictionary/Hashing approach.

### Comparison: The Two Approaches
#### 1. The SQL Solution (The "Pushdown")
This is the most efficient. We don't pull individual orders; we pull the final answer.

In [ ]:
-- We do the heavy lifting in the DB
-- This query turns 10,000,000 rows into maybe 500 rows of 'Abusers'
SELECT 
    user_id, 
    COUNT(order_id) as total_usage,
    SUM(discount_amount) as total_saved
FROM order_discounts
WHERE created_at >= datetime('now', '-24 hours')
GROUP BY user_id
HAVING COUNT(order_id) > 10; -- The 'Abuse' threshold

#### 2. The Python Solution (The "Tally" Logic)
If you must use Python (e.g., reading a massive log file), you use a dictionary to maintain "state" across chunks. This prevents the "split user" problem you identified.

In [ ]:
import pandas as pd

def process_massive_file(file_path):
    # A dictionary to keep track of EVERY user's count across all chunks
    user_tally = {} 
    chunk_size = 100000
    
    # Process the file in bites
    for chunk in pd.read_csv(file_path, chunksize=chunk_size):
        # We aggregate the chunk internally first
        chunk_counts = chunk.groupby('user_id')['order_id'].count()
        
        # Then we update our Global Tally
        for user_id, count in chunk_counts.items():
            user_tally[user_id] = user_tally.get(user_id, 0) + count
            
    # Finally, filter the dictionary for abusers
    abusers = {u: c for u, c in user_tally.items() if c > 10}
    return abusers

#### The 100x Takeaway
The SQL approach is faster and uses less network bandwidth.

The Python approach is more flexible but consumes more local memory (RAM) because the user_tally dictionary grows for every unique user in the dataset.